# Prepare and run MDAS Cancer notebooks in Google Colab

This separate notebook prepares an environment and runs the **original, unmodified** Classification or Survival notebook in a fresh isolated kernel. It does not add cells to, rewrite, or save over the source notebooks. All original training and advanced analysis cells run.

**Select a supported runtime first:** Runtime > Change runtime type > Runtime Version > **2026.07** (Python 3.12). A CPU runtime is sufficient. The default Python 3.13 runtime is incompatible with the published KaleCancer/PyKale requirements. If Python 3.12 is no longer offered, use the MDAS website's Binder option; this setup will stop instead of bypassing package constraints.

The isolated environment uses the repository's pinned CPU research requirements (including PyTorch 2.6). Colab's own PyTorch and kernel are left intact. Initial installation downloads the CPU ML stack and can take several minutes. Subsequent runs reuse the checkout and matching environment.

**Runtime isolation matters:** installing here does not prepare an original notebook opened independently in another Colab tab. This notebook executes the original from its repository checkout and offers a results download. The original Colab links remain useful for reading, but are not an automatically prepared interactive session. No Drive mount or uploaded files are required.

Use synthetic data only. Sessions are temporary: download your results before disconnecting.


In [ ]:
# Check the actual host-kernel Python, then obtain the public repository once.
from pathlib import Path
import subprocess
import sys

if sys.version_info[:2] != (3, 12):
    raise RuntimeError(
        f"This Colab kernel is Python {sys.version.split()[0]}. Select Runtime > Change runtime type > "
        "Runtime Version > 2026.07 (Python 3.12), reconnect, and run again. "
        "Installing another Python executable does not change this notebook's kernel."
    )

REPO = Path("/content/mdas-colab")
if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/pykale/mdas", str(REPO)], check=True)
else:
    remote = subprocess.check_output(["git", "-C", str(REPO), "remote", "get-url", "origin"], text=True).strip()
    if remote.rstrip("/").removesuffix(".git") != "https://github.com/pykale/mdas":
        raise RuntimeError("The existing preparation checkout has an unexpected origin. Use a fresh Colab session.")

REVISION = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("MDAS checkout:", REVISION)
if not (REPO / "colab" / "environment.py").is_file():
    raise RuntimeError("The external Colab runner is not present on this public revision. The reviewed colab/ files must be published together. Use Binder meanwhile.")
# main is resolved once by clone; reruns deliberately retain that exact checkout.


In [ ]:
# Install only in a separate virtual environment, never in the Colab host.
import runpy

environment = runpy.run_path(str(REPO / "colab" / "environment.py"))
RESEARCH_PYTHON = environment["prepare"]()


In [ ]:
# Choose a notebook. "both" uses a different fresh kernel for each notebook.
notebook = "classification" #@param ["classification", "survival", "both"]
subprocess.run([
    str(RESEARCH_PYTHON), str(REPO / "colab" / "run_notebooks.py"), notebook,
    "--root", str(REPO),
], check=True)


## Download results

The download includes the executed notebook copy, an HTML report, execution timings, and the source SHA-256. The original files remain untouched. An execution copy contains results; opening it in another Colab tab still does not transfer the prepared environment or kernel state. Select a different notebook above and rerun the execution cell to use the same installation with a fresh kernel.


In [ ]:
import zipfile
from google.colab import files

results = REPO / "outputs" / "colab"
archive = REPO / "outputs" / "mdas-colab-results.zip"
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(results.iterdir()):
        if path.is_file():
            bundle.write(path, path.name)
files.download(str(archive))
